Célula 1 - Carregamento com Alinhamento Dinâmico de Colunas

In [3]:
import pandas as pd
import numpy as np
import glob
import os

caminho_dnit = '../data/raw/dnit/*pa*.csv'
arquivos_dnit = glob.glob(caminho_dnit)

lista_dfs_dnit = []
for arq in arquivos_dnit:
    # Voltando para utf-8 que é o padrão correto do DNIT
    df_ano = pd.read_csv(arq, sep=';', encoding='utf-8', low_memory=False)
    
    # 1. Tudo em minúsculo e troca espaço por underline
    df_ano.columns = df_ano.columns.str.lower().str.replace(' ', '_')
    # 2. REMOVE ACENTOS (superfície vira superficie)
    df_ano.columns = df_ano.columns.str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
    
    lista_dfs_dnit.append(df_ano)

df_dnit = pd.concat(lista_dfs_dnit, ignore_index=True)
print(f"Base unificada do DNIT carregada! Colunas tratadas. Formato: {df_dnit.shape}")

Base unificada do DNIT carregada! Colunas tratadas. Formato: (46390, 17)


Célula 2 - Filtro Fino e Imputação da Série Temporal

In [4]:
# 1. Lista atualizada SEM ACENTOS (agora vai encontrar a superficie perfeitamente)
colunas_desejadas = [
    'id_malha', 'uf', 'contrato', 'ano', 'mes', 'rodovia', 'km', 'sentido',
    'km_inicial', 'km_final', 'num_faixas', 'superficie', 'data_aval.',
    'ip', 'ic', 'icm', 'icm_unificado'
]

colunas_presentes = [col for col in colunas_desejadas if col in df_dnit.columns]
df_dnit_filtrado = df_dnit[colunas_presentes].copy()

# 2. Garantir numéricos e ordenação
df_dnit_filtrado['km'] = pd.to_numeric(df_dnit_filtrado['km'].astype(str).str.replace(',', '.'), errors='coerce')
df_dnit_filtrado = df_dnit_filtrado.dropna(subset=['rodovia', 'km'])

df_dnit_filtrado['ano'] = pd.to_numeric(df_dnit_filtrado['ano'], errors='coerce')
df_dnit_filtrado['mes'] = pd.to_numeric(df_dnit_filtrado['mes'], errors='coerce')
df_dnit_filtrado = df_dnit_filtrado.sort_values(by=['rodovia', 'km', 'ano', 'mes'])

# 3. Imputação Categórica (Textos)
colunas_textos = ['superficie', 'sentido']
for col in colunas_textos:
    if col in df_dnit_filtrado.columns:
        df_dnit_filtrado[col] = df_dnit_filtrado.groupby(['rodovia', 'km'])[col].ffill()
        df_dnit_filtrado[col] = df_dnit_filtrado.groupby(['rodovia', 'km'])[col].bfill()
        df_dnit_filtrado[col] = df_dnit_filtrado[col].fillna('Não Mapeado')

# 4. Imputação Numérica
colunas_indices = ['ic', 'ip', 'icm', 'icm_unificado', 'num_faixas']
for col in colunas_indices:
    if col in df_dnit_filtrado.columns:
        df_dnit_filtrado[col] = df_dnit_filtrado[col].astype(str).str.replace(',', '.')
        df_dnit_filtrado[col] = pd.to_numeric(df_dnit_filtrado[col], errors='coerce')
        
        df_dnit_filtrado[col] = df_dnit_filtrado.groupby(['rodovia', 'km'])[col].transform(lambda x: x.interpolate(method='linear'))
        df_dnit_filtrado[col] = df_dnit_filtrado.groupby(['rodovia', 'km'])[col].bfill().ffill()

# 5. Exportação
df_dnit_final = df_dnit_filtrado.drop_duplicates(subset=['rodovia', 'km'])
import os
os.makedirs('../data/processed', exist_ok=True)
df_dnit_final.to_csv('../data/processed/dnit_limpo.csv', index=False, sep=';', encoding='utf-8')

print(f"Sucesso! A coluna 'superficie' está na base? {'superficie' in df_dnit_final.columns}")

Sucesso! A coluna 'superficie' está na base? True
